In [ ]:
import glob
import io
import itertools
import math
import os
import pickle
import subprocess
import warnings
from pathlib import Path
from statistics import median

import gzip
import boto3
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import norm
from supervenn import supervenn
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

%config InlineBackend.figure_format = 'png'
plt.rcParams["pdf.fonttype"] = "truetype"
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 200

sns.set_style("ticks")
sns.axes_style("white")

# Cohort assembly originally ran against an internal cohort database, configured
# here with an API key. The public pipeline reads prepared tables directly, so no
# client and no credential is needed.

import csv

import pyreadr
import rpy2.robjects as ro
from biomart import BiomartServer
from neuroCombat import neuroCombat
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import (
    BoolVector,
    DataFrame,
    FloatVector,
    IntVector,
    ListVector,
    Matrix,
    StrVector,
)
from scipy.interpolate import interp1d
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ── Font size constants (change FONT_BASE to scale all text globally) ──────
FONT_BASE   = 14
FONT_TITLE  = FONT_BASE
FONT_LABEL  = FONT_BASE
FONT_TICK   = FONT_BASE - 2
FONT_LEGEND = FONT_BASE - 2
FONT_ANNOT  = FONT_BASE - 2
FONT_SMALL  = FONT_BASE - 4
FONT_TINY   = FONT_BASE - 4

plt.rcParams.update({
    "svg.fonttype":     "none",
    "figure.dpi":       150,
    "font.size":        FONT_BASE,
    "axes.titlesize":   FONT_TITLE,
    "axes.labelsize":   FONT_LABEL,
    "xtick.labelsize":  FONT_TICK,
    "ytick.labelsize":  FONT_TICK,
    "legend.fontsize":  FONT_LEGEND,
    "figure.titlesize": FONT_TITLE,
})

In [ ]:
def log_transform_by_cohort(
    exp_df: pd.DataFrame,
    ann_df: pd.DataFrame,
    batch_col: str = "COHORT_LABEL",
    threshold: float = 30,
) -> pd.DataFrame:
    """
    Apply log2(x+1) transform per cohort only when max value exceeds threshold.

    Parameters
    ----------
    exp_df
        Expression matrix (samples × genes).
    ann_df
        Annotation with batch_col column.
    batch_col
        Column used to group samples (COHORT_LABEL, not RNA_BATCH).
    threshold
        Cohorts with max > threshold are log2(x+1) transformed.

    Returns
    -------
    Log2-transformed expression matrix.
    """
    batches = ann_df[batch_col].value_counts().index
    df_list = []
    for batch in batches:
        samples = ann_df.index[ann_df[batch_col] == batch]
        exp_local = exp_df.reindex(samples)
        if exp_local.max().max() > threshold:
            exp_local = np.log2(exp_local + 1)
        df_list.append(exp_local)
    return pd.concat(df_list, axis=0)

def plot_batch_distributions(exp_df, ann_df, batch_col='RNA_BATCH',
                              ncols=8, figsize=(28, 20), title_prefix=""):
    batches = ann_df[batch_col].value_counts().index
    nrows   = int(np.ceil(len(batches) / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for ax, batch in zip(axs.flatten(), batches):
        samples = ann_df.index[ann_df[batch_col] == batch]
        vals    = exp_df.reindex(samples).values#.ravel()
        vals    = vals[~np.isnan(vals)]
        ax.hist(vals, bins=40, density=True, alpha=0.7, color="steelblue")
        ax.set_title(batch, fontsize=10)
        #ax.set_xlim(-2, 18)
    for ax in axs.flatten()[len(batches):]:
        ax.set_visible(False)
    fig.suptitle(f"{title_prefix} Expression distributions per {batch_col}", fontsize=10)
    plt.tight_layout()
    return fig


In [ ]:
comb_ann

## Initial expressions and annotation download

In [ ]:
comb_exp = pd.read_csv(
    os.path.join(FL_DATA_ROOT, "comb_exp.tsv"), sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    "~/FL_harmonization/comb_ann_unified.csv"
).set_index("Unnamed: 0")

comb_ann["Diagnosis_with_coo"] = (
    comb_ann.Diagnosis_cell_type_general + "_" + comb_ann["coo_bg"].fillna("")
)
comb_ann["Diagnosis_unified_with_coo"] = (
    comb_ann.Diagnosis_cell_type_unified + "_" + comb_ann["coo_bg"].fillna("")
)
comb_ann.loc[comb_ann.index.isin(["PUB_Suntsova_GSE120795"]), "RNA_BATCH"] = (
    "RNASeq_FF_Unknown"
)

common_samples = comb_exp.index.intersection(comb_ann.index)  # [:7000]
comb_exp = comb_exp.loc[common_samples]  # .dropna(axis=1)
comb_exp = comb_exp.loc[~comb_exp.index.duplicated(keep="first"), :]
# Handle NaNs if necessary
comb_ann = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]


comb_ann

#### Imputed expressions download

In [ ]:
comb_exp = pd.read_csv(
    "~/FL_harmonization/shambhala_adoption/Shambhala2/large_tables/S0_no_removal__knn__exp.tsv", sep="\t"
).set_index("Unnamed: 0")
comb_ann = pd.read_csv(
    "~/FL_harmonization/comb_ann_unified.csv"
).set_index("Unnamed: 0")

comb_ann["Diagnosis_with_coo"] = (
    comb_ann.Diagnosis_cell_type_general + "_" + comb_ann["coo_bg"].fillna("")
)
comb_ann["Diagnosis_unified_with_coo"] = (
    comb_ann.Diagnosis_cell_type_unified + "_" + comb_ann["coo_bg"].fillna("")
)
comb_ann.loc[comb_ann.index.isin(["PUB_Suntsova_GSE120795"]), "RNA_BATCH"] = (
    "RNASeq_FF_Unknown"
)

common_samples = comb_exp.index.intersection(comb_ann.index)  # [:7000]
comb_exp = comb_exp.loc[common_samples]  # .dropna(axis=1)
comb_exp = comb_exp.loc[~comb_exp.index.duplicated(keep="first"), :]
# Handle NaNs if necessary
comb_ann = comb_ann.loc[~comb_ann.index.duplicated(keep="first"), :]


In [ ]:
comb_exp_log = log_transform_by_cohort(comb_exp, comb_ann)
comb_exp_log

In [ ]:
comb_exp_log.dropna(axis=0)[:1000].to_csv(
    "./Shambhala2/comb_exp_log_strict_subset_first_1000.csv",
    index=True,  # Keep the index (the gene names)
    #index_label="SYMBOL",  # Name the index column "SYMBOL" in the CSV header
    #quoting=csv.QUOTE_ALL,  # ⬅️ Force " " around everything
)

In [ ]:
comb_exp_log.dropna(axis=0)[:1000]

In [ ]:
comb_exp_log.dropna(axis=1)[:4000].T.to_csv(
    "./Shambhala2/comb_exp_log_dedup_strict_subset_first_4000.csv",
    index=True,  # Keep the index (the gene names)
    index_label="SYMBOL",  # Name the index column "SYMBOL" in the CSV header
    quoting=csv.QUOTE_ALL,  # ⬅️ Force " " around everything
)

In [ ]:
comb_exp_log.dropna(axis=1)[4000:].T.to_csv(
    "./Shambhala2/comb_exp_log_dedup_strict_subset_4000_and_beyond.csv",
    index=True,  # Keep the index (the gene names)
    index_label="SYMBOL",  # Name the index column "SYMBOL" in the CSV header
    quoting=csv.QUOTE_ALL,  # ⬅️ Force " " around everything
)

In [ ]:
comb_exp_log.dropna(axis=1).T.to_csv(
    "./Shambhala2/comb_exp_log_dedup_strict.csv",
    index=True,  # Keep the index (the gene names)
    index_label="SYMBOL",  # Name the index column "SYMBOL" in the CSV header
    quoting=csv.QUOTE_ALL,  # ⬅️ Force " " around everything
)

In [ ]:
comb_exp_log.T.to_csv(
    "./Shambhala2/comb_exp_log_dedup.csv",
    index=True,  # Keep the index (the gene names)
    index_label="SYMBOL",  # Name the index column "SYMBOL" in the CSV header
    quoting=csv.QUOTE_ALL,  # ⬅️ Force " " around everything
)

In [ ]:
sns.distplot(comb_exp.T["GSM4644254-PUB_DLBCL_CARTFARAMAND"])

## P and Q datasets

In [ ]:
q_dataset = pd.read_csv("./Shambhala2/Q0.csv") # GTEx, NGS
q_dataset.T[0].value_counts()

In [ ]:
p_dataset = pd.read_csv("./Shambhala2/P0.csv") # affymetrix, check
p_dataset.T[0].value_counts()

### Calibration datasets transformation for the containerized shambhala

In [ ]:
q_dataset.columns

In [ ]:
q_dataset = pd.read_csv("./Shambhala2/Q0.csv")
q_dataset = q_dataset.set_index('SYMBOL')
q_dataset = q_dataset[q_dataset.index != 'Sample_type'].T
q_dataset.columns.name = ''
q_dataset.to_csv('./Calibration_datasets/Q0_standard.csv')

In [ ]:
q_dataset

In [ ]:
p_dataset

In [ ]:
p_dataset = pd.read_csv("./Shambhala2/P0.csv")
p_dataset = p_dataset.set_index('SYMBOL')
p_dataset = p_dataset[p_dataset.index != 'Sample_type'].T
p_dataset.columns.name = ''
p_dataset.to_csv('./Calibration_datasets/P0_standard.csv')

In [ ]:
comb_exp_log[comb_ann['TUMOR_NORMAL'] == 'Normal'].to_csv('./Calibration_datasets/Normal_B_cells_extended.csv')


In [ ]:
comb_exp_log[comb_ann['Major_group'] == 'Normal_B_cells'].to_csv('./Calibration_datasets/Normal_B_legacy.csv')


In [ ]:
comb_exp_log[comb_ann['Major_group'] == 'Kassandra'].to_csv('./Calibration_datasets/Normal_B_Kassandra.csv')


In [ ]:
comb_exp_log[(comb_ann['TUMOR_NORMAL'] == 'Normal') & (comb_ann['PLATFORM_RNA'] == 'RNASeq')].to_csv('./Calibration_datasets/Normal_B_RNASeq.csv')


In [ ]:
comb_exp_log[(comb_ann['TUMOR_NORMAL'] == 'Normal') & (comb_ann['PLATFORM_RNA'] == 'GPL570')].to_csv('./Calibration_datasets/Normal_B_GPL570.csv')


In [ ]:
comb_exp_log[comb_ann['TUMOR_NORMAL'] == 'Normal']

In [ ]:
list(ante.columns)

In [ ]:
ante = pd.read_csv('./Calibration_datasets/raw/ANTE.csv')

ante = ante.set_index('SYMBOL')
ante = ante[ante.index != 'Sample_type'].T
ante.columns.name = ''
ante.to_csv('./Calibration_datasets/ANTE.csv')

ante

In [ ]:
shambhala_attempt =  pd.read_csv('Shambhala2/large_tables/harmonized_subset_260515.csv')
shambhala_attempt

In [ ]:
GTExAffymetrix = pd.read_csv('./Calibration_datasets/raw/GTExAffymetrix.csv')
GTExAffymetrix

In [ ]:
GTExAffymetrix = pd.read_csv('./Calibration_datasets/raw/GTExAffymetrix.csv')

GTExAffymetrix = GTExAffymetrix.set_index('SYMBOL')
GTExAffymetrix= GTExAffymetrix[GTExAffymetrix.index != 'Sample_type'].T
GTExAffymetrix.columns.name = ''
GTExAffymetrix.to_csv('./Calibration_datasets/GTExAffymetrix.csv')

GTExAffymetrix

In [ ]:
test3 = pd.read_excel('./Calibration_datasets/raw/Test3.xlsx')
test3

In [ ]:
test3.min().min()

In [ ]:
test3.T[0].value_counts()

In [ ]:
comb_ann[comb_ann['Major_group'] == 'Normal_B_cells'].PLATFORM_RNA.value_counts()

In [ ]:
comb_ann[comb_ann['Major_group'] == 'Kassandra'].PLATFORM_RNA.value_counts()

In [ ]:
comb_ann[comb_ann['TUMOR_NORMAL'] == 'Normal'].PLATFORM_RNA.value_counts()

## Python wrapper for R script Shambhala

In [ ]:
import os

import boto3
import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# 1. Enable Pandas to R DataFrame conversion
pandas2ri.activate()

# 2. Load the R environment
base = importr("base")
utils = importr("utils")

# Define paths strictly in your user space
PROJECT_DIR = os.path.expanduser(
    "~/FL_harmonization/shambhala_adoption/"
)
SHAMBHALA_DIR = os.path.join(PROJECT_DIR, "Shambhala2")
shambhala_r_path = os.path.join(SHAMBHALA_DIR, "Shambhala2.R")

P_REF = os.path.join(SHAMBHALA_DIR, "P0.csv")
Q_REF = os.path.join(SHAMBHALA_DIR, "Q0.csv")

# 3. CRITICAL: Change Python's working directory so Octave finds the .m files!
os.chdir(SHAMBHALA_DIR)
print(f"📂 Changed working directory to: {os.getcwd()}")

# Source the patched Shambhala R script containing 'Shambhala2_flex'
print("📥 Sourcing Shambhala R script...")
robjects.r(f'source("{shambhala_r_path}")')


# 4. Define the Execution Wrapper (Updated for In-Memory DataFrame Processing)
def run_shambhala_harmonization(expression_df, p_dataset_path, q_dataset_path):
    print("🔄 Processing and cleaning Python DataFrame...")

    # 1. Format the DataFrame to match Shambhala's strict expectations
    temp_df = expression_df.copy()

    # Handle the "Unnamed: 0" column if it exists, or index reset
    if "Unnamed: 0" in temp_df.columns:
        temp_df.rename(columns={"Unnamed: 0": "SYMBOL"}, inplace=True)
    elif "SYMBOL" not in temp_df.columns:
        temp_df = temp_df.reset_index()
        temp_df.rename(columns={temp_df.columns[0]: "SYMBOL"}, inplace=True)

    # 2. Clean NaN values (Critical for Octave matrix math)
    # Drop rows where ALL expression values are NaN (ignoring the SYMBOL column)
    sample_cols = [col for col in temp_df.columns if col != "SYMBOL"]
    initial_rows = len(temp_df)
    temp_df.dropna(subset=sample_cols, how="all", inplace=True)

    # Fill remaining isolated NaNs with 0 (as Shambhala is log-transformed expression data)
    # temp_df.fillna(0, inplace=True)

    print(
        f"🧹 Data cleaned: Dropped {initial_rows - len(temp_df)} completely empty rows."
    )

    print("🔄 Converting Python DataFrame to R DataFrame...")
    # Convert Pandas DataFrame directly to R DataFrame
    r_dataframe = pandas2ri.py2rpy(temp_df)

    print("🚀 Triggering Shambhala2_flex (In-Memory R -> Octave pipeline)...")

    # CRITICAL: Call the newly written Shambhala2_flex function
    shambhala_func = robjects.globalenv["Shambhala2_flex"]

    try:
        # Pass the in-memory R dataframe directly, skipping CSV I/O
        harmonized_r_matrix = shambhala_func(
            r_dataframe,
            p_dataset_path,
            q_dataset_path,
            True,  # delete_buffer_files
            5,  # k clusters
        )

        # Convert the result back to a Python Pandas DataFrame
        harmonized_df = pandas2ri.rpy2py(harmonized_r_matrix)

        print("✅ Harmonization complete!")
        return harmonized_df

    except Exception as e:
        raise Exception(f"❌ Shambhala execution failed. Check logs: {e}")

In [ ]:
harmonized_exp = run_shambhala_harmonization(comb_exp.T, P_REF, Q_REF)

In [ ]:
harmonized_exp

In [ ]:
if __name__ == "__main__":
    S3_BUCKET = "fl-disser-data"
    S3_KEY = "raw_matrices/cohort_concat_unnormalized.csv"

    # Reference datasets (P and Q) must exist in the cloned repo or be generated
    P_REF = os.path.join(SHAMBHALA_DIR, "P_dataset.csv")
    Q_REF = os.path.join(SHAMBHALA_DIR, "Q_dataset.csv")

    raw_exp = fetch_data_from_aws(S3_BUCKET, S3_KEY)
    harmonized_exp = run_shambhala_harmonization(raw_exp, P_REF, Q_REF)

    if harmonized_exp is not None:
        save_path = os.path.join(PROJECT_DIR, "shambhala_harmonized_fl_cohorts.csv")
        harmonized_exp.to_csv(save_path)
        print(f"💾 Saved to {save_path}")

## Shambhala harmonized dataset upload

In [ ]:
shambhala_1 = pd.read_csv('./Shambhala2/shambhala_harmonized_strict_first_4000.csv').set_index('SYMBOL')
shambhala_2 = pd.read_csv('./Shambhala2/shambhala_harmonized_strict_subset_4000_and_beyond.csv').set_index('SYMBOL')
shambhala_harmonized = pd.concat([shambhala_1, shambhala_2], axis=1).T
print(len(list(set(shambhala_harmonized.index) & set(comb_ann.index))))
shambhala_harmonized.index = shambhala_harmonized.index.str.replace('.', '-')
print(len(list(set(shambhala_harmonized.index) & set(comb_ann.index))))
comb_ann_shambhala = comb_ann.reindex(shambhala_harmonized.index)
shambhala_harmonized

In [ ]:
comb_ann.index[comb_ann.index.str.startswith('GSM3302930')]

In [ ]:
shambhala_harmonized.index

## PCA UMAP

### Helper plotting functions

In [ ]:
def compute_pca_coords(
    exp: pd.DataFrame, n_components: int = 50
) -> tuple[np.ndarray, PCA]:
    """Standardise then PCA; returns (coords, pca_obj)."""
    from sklearn.preprocessing import StandardScaler
    X = StandardScaler().fit_transform(exp.dropna(axis=1).values)
    pca = PCA(n_components=min(n_components, min(X.shape) - 1))
    coords = pca.fit_transform(X)
    return coords, pca


def compute_umap_coords(
    exp: pd.DataFrame,
    n_pcs: int = 50,
    n_neighbors: int = 30,
    min_dist: float = 0.3,
    random_state: int = 42,
) -> np.ndarray:
    """UMAP on top-N PCA coordinates."""
    import umap as umap_lib
    pca_coords, _ = compute_pca_coords(exp.dropna(axis=1), n_components=n_pcs)
    reducer = umap_lib.UMAP(
        n_neighbors=n_neighbors, min_dist=min_dist,
        n_components=2, random_state=random_state,
    )
    return reducer.fit_transform(pca_coords)


def compute_tsne_coords(
    exp: pd.DataFrame,
    n_pcs: int = 50,
    perplexity: float = 30.0,
    random_state: int = 42,
) -> np.ndarray:
    """t-SNE on top-N PCA coordinates."""
    from sklearn.manifold import TSNE
    pca_coords, _ = compute_pca_coords(exp.dropna(axis=1), n_components=n_pcs)
    return TSNE(n_components=2, perplexity=perplexity, random_state=random_state).fit_transform(pca_coords)

def pca_plot(
    coords: np.ndarray,
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    ax: plt.Axes,
    title: str = "",
    xlabel: str = "PC1",
    ylabel: str = "PC2",
    s: float = 3,
    alpha: float = 0.5,
    prefix_match: bool = False,
    rasterized: bool = True,
    pca_obj: PCA | None = None,
) -> None:
    """
    Draw a single scatter plot (PCA / UMAP / tSNE) on ax.

    Parameters
    ----------
    coords : np.ndarray  shape (n_samples, 2+)
    ann_df : pd.DataFrame  annotation aligned to coords rows
    color_col : str  annotation column used for colour mapping
    palette : dict  colour dictionary
    ax : plt.Axes
    pca_obj : PCA or None  if given, overrides xlabel/ylabel with variance %
    """
    if pca_obj is not None:
        v1 = pca_obj.explained_variance_ratio_[0] * 100
        v2 = pca_obj.explained_variance_ratio_[1] * 100
        xlabel = f"PC1 ({v1:.1f}%)"
        ylabel = f"PC2 ({v2:.1f}%)"
    colors = _color_by_col(ann_df, color_col, palette, prefix_match=prefix_match)
    ax.scatter(
        coords[:, 0], coords[:, 1],
        c=colors.values, s=s, alpha=alpha, linewidths=0, rasterized=rasterized,
    )
    ax.set_xlabel(xlabel, fontsize=FONT_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONT_LABEL)
    ax.tick_params(labelsize=FONT_TICK)
    if title:
        ax.set_title(title, fontsize=FONT_TITLE)


def umap_plot(
    coords: np.ndarray,
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    ax: plt.Axes,
    title: str = "",
    s: float = 3,
    alpha: float = 0.5,
    prefix_match: bool = False,
    rasterized: bool = True,
    pca_obj: PCA | None = None,
) -> None:
    """Draw a single UMAP scatter (delegates to pca_plot with UMAP labels)."""
    pca_plot(
        coords, ann_df, color_col, palette, ax,
        title=title, xlabel="UMAP1", ylabel="UMAP2",
        s=s, alpha=alpha, prefix_match=prefix_match, rasterized=rasterized,
    )


def tsne_plot(
    coords: np.ndarray,
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    ax: plt.Axes,
    title: str = "",
    s: float = 3,
    alpha: float = 0.5,
    prefix_match: bool = False,
    rasterized: bool = True,
    pca_obj: PCA | None = None,
) -> None:
    """Draw a single tSNE scatter (delegates to pca_plot with tSNE labels)."""
    pca_plot(
        coords, ann_df, color_col, palette, ax,
        title=title, xlabel="tSNE1", ylabel="tSNE2",
        s=s, alpha=alpha, prefix_match=prefix_match, rasterized=rasterized,
    )

def _color_by_col(
    ann_df: pd.DataFrame,
    color_col: str,
    palette: dict,
    prefix_match: bool = False,
) -> pd.Series:
    """Map an annotation column to hex colors via a palette dict."""
    if not prefix_match:
        return ann_df[color_col].map(palette).fillna("#AAAAAA")

    def _match(label: str) -> str:
        if pd.isna(label):
            return "#AAAAAA"
        for prefix, color in palette.items():
            if str(label).startswith(str(prefix)):
                return color
        return "#AAAAAA"

    return ann_df[color_col].apply(_match)


### Palettes

In [ ]:
# ── §5.0 Canonical project palettes (from all_cohorts_assembly.ipynb) ─────────
# Definitive color mappings for the entire dissertation.
# Use for all sample-level scatter plots in §5.

lymphoma_ontogeny_palette = {
    "Bone_marrow_CD19+": "blue",
    "Immature": "#08519c",
    "Naive": "#4292c6",
    "GC": "#006d2c",
    "Centroblast": "#41ab5d",
    "Centrocyte": "#a1d99b",
    "B_cells": "lawngreen",
    "MZ": "olive",
    "Memory": "#00ced1",
    "Plasmablast": "#5f9ea0",
    "Plasma": "grey",
    "Follicular_Lymphoma": "#6a0dad",
    "Follicular_Lymphoma_GCB": "#9400d3",
    "Diffuse_Large_B_Cell_Lymphoma_ABC": "#ff4500",
    "Diffuse_Large_B_Cell_Lymphoma_GCB": "#d62728",
    "Diffuse_Large_B_Cell_Lymphoma": "#8b0000",
    "High_Grade_B_Cell_Lymphoma": "#ff00ff",
    "Burkitt_Lymphoma": "orange",
}

lymphoma_ontogeny_palette_detailed = {
    # --- NORMAL B-CELLS: THE COLD SPECTRUM (Blues, Teals, Greens) ---
    # Primary/Early Development
    'Bone_marrow_CD19+_': 'blue',    # Deepest Navy
    'Immature_': '#08519c',             # Dark Blue
    'Naive_': '#4292c6',                # Medium Blue
    
    # Germinal Center (Cold variants)
    'GC_': '#006d2c',                   # Dark Green
    'Centroblast_': '#41ab5d',          # Medium Green
    'Centrocyte_': '#a1d99b',           # Pale Green
    
    # Mature & Memory
    'B_cells_': 'lawngreen',              # Forest Green
    'MZ_': 'olive',                   # Seafoam Green
    'Memory_': '#00ced1',               # Dark Turquoise
    
    # Terminal Differentiation (Cold Neutrals)
    'Plasmablast_': '#5f9ea0',          # Cadet Blue
    'Plasma_': 'grey',               # Steel Blue-Gray

    # --- MALIGNANT TYPES: THE WARM SPECTRUM (Reds, Oranges, Purples, Black) ---
    # Follicular (The Purple/Magenta family)
    'Follicular_Lymphoma_': '#6a0dad',      # Vivid Purple
    'Follicular_Lymphoma_GCB': '#9400d3',   # Dark Violet
    
    # DLBCL (The Red/Orange family)
    'Diffuse_Large_B_Cell_Lymphoma_ABC': '#ff4500', # Orange-Red
    'Diffuse_Large_B_Cell_Lymphoma_GCB': '#d62728', # Classic Red
    'Diffuse_Large_B_Cell_Lymphoma_': '#8b0000',    # Dark Red
    
    # High-Grade & Burkitt (Deep Contrast)
    'High_Grade_B_Cell_Lymphoma_': '#ff00ff',       # Magenta/Fuchsia
    'Burkitt_Lymphoma_': 'orange'                  # Absolute Black
}

rna_batch_palette = {
    "RNASeq_FF_Total": "#1f77b4",
    "RNASeq_FF_PolyA": "#17becf",
    "RNASeq_FFPE_Exome_capture": "#9467bd",
    "RNASeq_FF_rRNADepletion": "#000080",
    "RNASeq_FFPE_PolyA": "#8c564b",
    "RNASeq_FF_Unknown": "#bcbd22",
    "RNASeq_FF_Exome_capture": "#2ca02c",
    "GPL96+97_FF_Unknown": "#d62728",
    "GPL570_FF_Unknown": "#ff7f0e",
    "GPL570_Unknown_Unknown": "#ffbb78",
    "GPL570_FFPE_Unknown": "#dbdb8d",
    "GPL96_FF_Unknown": "#f7b6d2",
    "GPL96_FFPE_Unknown": "#e377c2",
    "GPL6244_FF_Unknown": "#e9967a",
    "GPL6244_FFPE_Unknown": "#fa8072",
    "GPL13938_FFPE_Unknown": "#800000",
    "GPL1708_FF_Unknown": "#7f7f7f",
    "GPL14951_FFPE_Unknown": "#c49c94",
    "GPL17077_FF_Unknown": "#c5b0d5",
    "GPL13158_FF_Unknown": "#ad494a",
    "GPL17586_FF_Unknown": "#008b8b",
    "GPL17047_FF_Unknown": "#4682b4",
    "GPL887_FFPE_Unknown": "#b0c4de",
    "GPL10739_FF_Unknown": "#9edae5",
    "GPL8432_FFPE_Unknown": "#006400",
    "GPL20188_FF_Unknown": "#556b2f",
    "GPL23541_FF_Unknown": "#8fbc8f",
    "GPL16686_FF_Unknown": "#333333",
    "GPL26356_FF_Unknown": "#a0522d",
}

platform_palette = {
    "RNASeq": "#4B0082",
    "RNAseq": "#4B0082",
    "Kassandra": "#6A5ACD",
    "GPL96+97": "#FF4500",
    "GPL96+GPL97": "#FF4500",
    "GPL570": "#FF8C00",
    "GPL96": "#B22222",
    "GPL6244": "#E9967A",
    "GPL13938": "#CD5C5C",
    "A-GEOD-19803": "#A52A2A",
    "GPL1708": "#008080",
    "GPL14951": "#4682B4",
    "GPL17077": "#00CED1",
    "GPL13158": "#1E90FF",
    "GPL17586": "#20B2AA",
    "GPL17047": "#5F9EA0",
    "GPL887": "#B0C4DE",
    "GPL10739": "#2E8B57",
    "GPL10739+GPL19251": "#3CB371",
    "GPL20188": "#6B8E23",
    "GPL23541": "#8FBC8F",
    "GPL8432": "#006400",
    "GPL16686": "#8B4513",
    "GPL26356": "#A0522D",
    float("nan"): "#D3D3D3",
    None: "#D3D3D3",
}

cohort_palette = {
    "PUB_DLBCL_CARTFARAMAND": "#f87d06",
    "PUB_DLBCL_JIANGGSE147986": "#1482b7",
    "PUB_DLBCL_GOYA": "#b72114",
    "PUB_DLBCL_NCICCR": "#b7b014",
    "PUB_DLBCL_PARKSRP100394": "#5bd66f",
    "PUB_DLBCL_SHISRP312586": "#2306f8",
    "PUB_DLBCL_YANCART": "#d67e5b",
    "PUB_FL_DAVE": "#0626f8",
    "PUB_FL_GSE62241": "#b71481",
    "PUB_FL_HUET": "#14b75d",
    "PUB_FL_PARSA": "#146db7",
    "PUB_FL_GUO": "#a3d65b",
    "PUB_FL_GSE148070": "#5b95d6",
    "PUB_FL_BRODTKORB": "#3514b7",
    "PUB_FL_CIBERSORTX": "#9ff806",
    "PUB_FL_HORN": "#5bd6cb",
    "PUB_FL_PASTORE": "#d66f5b",
    "PUB_FL_LEBRUN": "#b71496",
    "PUB_FL_ORICCHIO": "#9d5bd6",
    "PUB_FL_TASKINEN": "#14b772",
    "PUB_FL_DREYLING": "#8d5bd6",
    "PUB_FL_NCIFFPEREBUILT": "#f80625",
    "PUB_FL_NCISTAUDT": "#f80682",
    "PUB_FL_TOBIN": "#069ef8",
    "PUB_DLBCL_GSE11318": "#f80644",
    "PUB_DLBCL_BAGS": "#5bd6ac",
    "PUB_DLBCL_ETABM346": "#5bd660",
    "PUB_DLBCL_DLC1": "#5bd6bb",
    "PUB_DLBCL_NCIACALA": "#06f860",
    "PUB_DLBCL_GSE10846": "#5b85d6",
    "PUB_DLBCL_GSE117556": "#4914b7",
    "PUB_DLBCL_GSE19246": "#9b06f8",
    "PUB_DLBCL_GSE22898": "#c2d65b",
    "PUB_DLBCL_GSE23501": "#bef806",
    "PUB_DLBCL_GSE31312": "#b73614",
    "PUB_DLBCL_GSE32918": "#f706f8",
    "PUB_DLBCL_GSE34171": "#5e14b7",
    "PUB_DLBCL_GSE64555": "#d65bb3",
    "PUB_DLBCL_GSE87371": "#06dbf8",
    "PUB_DLBCL_GSE93984": "#d65ba3",
    "PUB_DLBCL_LYMPHOMICS": "#d65b84",
    "COLL_DLBCL_UMLOSSOSTRANSL": "#5b75d6",
    "CRON3": "#06f8f6",
    "E-MTAB-3974": "#f806da",
    "GSE107683": "#5bd1d6",
    "GSE12195": "#f89c06",
    "GSE12366": "#0607f8",
    "GSE12453": "#b2d65b",
    "GSE12845": "#66d65b",
    "GSE13411": "#14b787",
    "GSE136248": "#4206f8",
    "GSE136249": "#1bb714",
    "GSE141005": "#b906f8",
    "GSE144089": "#f80663",
    "GSE149089": "#f85e06",
    "GSE15271": "#aab714",
    "GSE156751": "#14b748",
    "GSE17186": "#1458b7",
    "GSE19599": "#b78614",
    "GSE22886": "#d806f8",
    "GSE36907": "#bc5bd6",
    "GSE38697": "#b79b14",
    "GSE46261": "#45b714",
    "GSE51528": "#1444b7",
    "GSE55267": "#1497b7",
    "GSE56314": "#b714ab",
    "GSE61912": "#76d65b",
    "GSE64028": "#06f841",
    "GSE68878": "#d68e5b",
    "GSE69033": "#06f822",
    "GSE85543": "#d2d65b",
    "GSE89282": "#06bdf8",
    "GSE99635": "#30b714",
    "Leandro-Normals": "#af14b7",
    "PUB_Beguelin_GSE45982": "#d6ca5b",
    "PUB_Health_B_cells_GSE110669": "#d69e5b",
    "PUB_Health_B_cells_GSE111310": "#b7141c",
    "PUB_Health_B_cells_GSE115655": "#b71431",
    "PUB_Health_B_cells_GSE119234": "#06f87f",
    "PUB_Health_B_cells_GSE139833": "#59b714",
    "PUB_Health_B_cells_GSE154583": "#5b66d6",
    "PUB_Health_B_cells_GSE92387": "#f8bb06",
    "PUB_Health_Bunting_GSE84022": "#d65b94",
    "PUB_Pastore_GSE119103": "#ac5bd6",
    "PUB_Suntsova_GSE120795": "#80f806",
    "Kassandra": "#14b79c",
    "SOM": "#61f806",
    "MDA_Strati_FL": "#2014b7",
}

### Plotting

In [ ]:
pca_coords = compute_pca_coords(shambhala_harmonized)[0]
pca_coords

In [ ]:
umap_coords = compute_umap_coords(shambhala_harmonized)
tsne_coords = compute_tsne_coords(shambhala_harmonized)

In [ ]:
cohort_palette

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True, squeeze=False)

pca_plot(coords=pca_coords, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL", xlabel = "PC1", ylabel = "PC2")


In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    squeeze=False,
)

umap_plot(coords=umap_coords, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH")
umap_plot(coords=umap_coords, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology")
umap_plot(coords=umap_coords, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform")
umap_plot(coords=umap_coords, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed")
umap_plot(coords=umap_coords, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL")
umap_plot(coords=umap_coords, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL")


In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    squeeze=False,
)

tsne_plot(coords=tsne_coords, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH")
tsne_plot(coords=tsne_coords, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology")
tsne_plot(coords=tsne_coords, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform")
tsne_plot(coords=tsne_coords, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed")
tsne_plot(coords=tsne_coords, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL")
tsne_plot(coords=tsne_coords, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL")


In [ ]:



fig, ax = plt.subplots(figsize=(5, 5))
pca_plot(
    coords=pca_coords,
    ann_df=comb_ann,
    color_col = 'RNA_BATCH',
    palette=rna_batch_palette,
    ax=ax,
    title = "Shambhala PCA by RNA_BATCH",
    xlabel = "PC1",
    ylabel = "PC2",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

fig, ax = plt.subplots(figsize=(5, 5))
pca_plot(
    coords=pca_coords,
    ann_df=comb_ann,
    color_col = 'Diagnosis_cell_type_unified',  
    palette=lymphoma_ontogeny_palette,
    ax=ax,
    title = "Shambhala PCA by biology",
    xlabel = "PC1",
    ylabel = "PC2",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

fig, ax = plt.subplots(figsize=(5, 5))
pca_plot(
    coords=pca_coords,
    ann_df=comb_ann,
    color_col = 'PLATFORM_RNA',  
    palette=platform_palette,
    ax=ax,
    title = "Shambhala PCA by platform",
    xlabel = "PC1",
    ylabel = "PC2",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

fig, ax = plt.subplots(figsize=(5, 5))
pca_plot(
    coords=pca_coords,
    ann_df=comb_ann,
    color_col = 'Diagnosis_unified_with_coo',  
    palette=lymphoma_ontogeny_palette_detailed,
    ax=ax,
    title = "Shambhala PCA by biology detailed",
    xlabel = "PC1",
    ylabel = "PC2",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
umap_plot(
    coords=umap_coords,
    ann_df=comb_ann,
    color_col = 'RNA_BATCH',
    palette=rna_batch_palette,
    ax=ax,
    title = "Shambhala UMAP by RNA_BATCH",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

fig, ax = plt.subplots(figsize=(5, 5))
umap_plot(
    coords=umap_coords,
    ann_df=comb_ann,
    color_col = 'Diagnosis_cell_type_unified',  
    palette=lymphoma_ontogeny_palette,
    ax=ax,
    title = "Shambhala UMAP by biology",

    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

fig, ax = plt.subplots(figsize=(5, 5))
umap_plot(
    coords=umap_coords,
    ann_df=comb_ann,
    color_col = 'PLATFORM_RNA',  
    palette=platform_palette,
    ax=ax,
    title = "Shambhala tSNE by platform",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
tsne_plot(
    coords=tsne_coords,
    ann_df=comb_ann,
    color_col = 'RNA_BATCH',
    palette=rna_batch_palette,
    ax=ax,
    title = "Shambhala tSNE by RNA_BATCH",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

fig, ax = plt.subplots(figsize=(5, 5))
tsne_plot(
    coords=tsne_coords,
    ann_df=comb_ann,
    color_col = 'Diagnosis_cell_type_unified',  
    palette=lymphoma_ontogeny_palette,
    ax=ax,
    title = "Shambhala tSNE by biology",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

fig, ax = plt.subplots(figsize=(5, 5))
tsne_plot(
    coords=tsne_coords,
    ann_df=comb_ann,
    color_col = 'PLATFORM_RNA',  
    palette=platform_palette,
    ax=ax,
    title = "Shambhala tSNE by platform",
    s = 3,
    alpha = 0.5,
    prefix_match = False,
    rasterized = True
)

## Distributions plotting

In [ ]:
shambhala_harmonized

In [ ]:
# ── 2c. Per-batch expression distribution check (Strategy A) ─────────────────

shambhala_harmonized_log = log_transform_by_cohort(shambhala_harmonized, comb_ann)
plot_batch_distributions(shambhala_harmonized_log, comb_ann, batch_col='COHORT_LABEL', title_prefix="Shambhala for all",
                        figsize=(28, 20))
plt.show()

In [ ]:
comb_ann.reindex(shambhala_1.columns)

In [ ]:
comb_ann.reindex(shambhala_2.columns)

In [ ]:
plot_batch_distributions(shambhala_2.T, comb_ann.reindex(shambhala_2.columns), batch_col='COHORT_LABEL', title_prefix="Shambhala for all",
                        figsize=(28, 20))


# Analyze speeded-up and the original Shambhala harmonization for strict S0 P0, Q0 pair of datasets

Speed-up s3://$FL_S3_BUCKET/FL_batch_correction/exp/S0_no_removal__strict__shambhala_P0std_Q0std_speed_up__post0.tsv.gz

Original s3://$FL_S3_BUCKET/FL_batch_correction/exp/S0_no_removal__strict__shambhala_P0std_Q0std__post0.tsv.gz

## Files read

In [ ]:
s3 = boto3.client("s3")
body = s3.get_object(Bucket='$FL_S3_BUCKET', Key='FL_batch_correction/exp/S0_no_removal__strict__shambhala_P0std_Q0std_speed_up__post0.tsv.gz')["Body"].read()

speed_shambhala = pd.read_csv(io.BytesIO(body), sep=",", index_col=0)
speed_shambhala


In [ ]:
comb_ann_shambhala = comb_ann.reindex(speed_shambhala.index)

In [ ]:
s3 = boto3.client("s3")
body = s3.get_object(Bucket='$FL_S3_BUCKET', Key='FL_batch_correction/exp/S0_no_removal__strict__shambhala_P0std_Q0std__post0.tsv.gz')["Body"].read()

original_shambhala = pd.read_csv(io.BytesIO(gzip.decompress(body)), sep="\t", index_col=0)
original_shambhala


## PCA, UMAP, TSNE

In [ ]:
pca_coords_speed = compute_pca_coords(speed_shambhala)[0]
pca_coords_original = compute_pca_coords(original_shambhala)[0]
pca_coords_speed

In [ ]:
umap_coords_speed = compute_umap_coords(speed_shambhala)
tsne_coords_speed = compute_tsne_coords(speed_shambhala)
umap_coords_original = compute_umap_coords(original_shambhala)
tsne_coords_original = compute_tsne_coords(original_shambhala)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True, squeeze=False)

pca_plot(coords=pca_coords_original, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_original, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_original, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_original, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_original, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_original, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL", xlabel = "PC1", ylabel = "PC2")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True, squeeze=False)

pca_plot(coords=pca_coords_speed, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_speed, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_speed, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_speed, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_speed, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL", xlabel = "PC1", ylabel = "PC2")
pca_plot(coords=pca_coords_speed, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL", xlabel = "PC1", ylabel = "PC2")


In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    squeeze=False,
)

umap_plot(coords=umap_coords_original, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH")
umap_plot(coords=umap_coords_original, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology")
umap_plot(coords=umap_coords_original, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform")
umap_plot(coords=umap_coords_original, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed")
umap_plot(coords=umap_coords_original, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL")
umap_plot(coords=umap_coords_original, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL")


In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    squeeze=False,
)

umap_plot(coords=umap_coords_speed, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH")
umap_plot(coords=umap_coords_speed, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology")
umap_plot(coords=umap_coords_speed, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform")
umap_plot(coords=umap_coords_speed, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed")
umap_plot(coords=umap_coords_speed, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL")
umap_plot(coords=umap_coords_speed, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL")


In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    squeeze=False,
)

tsne_plot(coords=tsne_coords_original, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH")
tsne_plot(coords=tsne_coords_original, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology")
tsne_plot(coords=tsne_coords_original, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform")
tsne_plot(coords=tsne_coords_original, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed")
tsne_plot(coords=tsne_coords_original, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL")
tsne_plot(coords=tsne_coords_original, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL")


In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 10),
    sharex=True,
    sharey=True,
    squeeze=False,
)

tsne_plot(coords=tsne_coords_speed, ann_df=comb_ann_shambhala, color_col = 'RNA_BATCH', palette=rna_batch_palette, ax=axes[0, 0], title = "Shambhala PCA by RNA_BATCH")
tsne_plot(coords=tsne_coords_speed, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_cell_type_unified', palette=lymphoma_ontogeny_palette, ax=axes[1, 0], title = "Shambhala PCA by biology")
tsne_plot(coords=tsne_coords_speed, ann_df=comb_ann_shambhala, color_col = 'PLATFORM_RNA', palette=platform_palette, ax=axes[0, 1], title = "Shambhala PCA by Platform")
tsne_plot(coords=tsne_coords_speed, ann_df=comb_ann_shambhala, color_col = 'Diagnosis_unified_with_coo', palette=lymphoma_ontogeny_palette_detailed, ax=axes[1, 1], title = "Shambhala PCA by Biology detailed")
tsne_plot(coords=tsne_coords_speed, ann_df=comb_ann_shambhala, color_col = 'COHORT_LABEL', palette=cohort_palette, ax=axes[0, 2], title = "Shambhala PCA by COHORT_LABEL")
tsne_plot(coords=tsne_coords_speed, ann_df=comb_ann_shambhala, color_col = 'TUMOR_NORMAL', palette={'Tumor': 'red', 'Normal': 'green'}, ax=axes[1, 2], title = "Shambhala PCA by TUMOR_NORMAL")


## Mean relative difference and other metrics calculation

In [ ]:
relative_difference

In [ ]:
relative_difference = (original_shambhala - speed_shambhala).abs() / original_shambhala

In [ ]:
mean_rel_diff_cohort = relative_difference.merge(comb_ann_shambhala['COHORT_LABEL'], left_index=True, right_index=True).groupby('COHORT_LABEL').mean().mean(axis=1)
mean_rel_diff_cohort


In [ ]:

fig, ax = plt.subplots(figsize=(18,4))
sns.barplot(mean_rel_diff_cohort.sort_values()[::-1][1:])
plt.xticks(rotation=90)              

In [ ]:
fig, ax = plt.subplots(figsize=(18,4))
sns.barplot(comb_ann_shambhala['COHORT_LABEL'].value_counts().reindex(mean_rel_diff_cohort.sort_values()[::-1][1:].index))
plt.xticks(rotation=90) 

In [ ]:
original_shambhala.mean()

In [ ]:
mean_rel_gene_change = relative_difference.merge(comb_ann_shambhala['COHORT_LABEL'], left_index=True, right_index=True).groupby('COHORT_LABEL').mean().mean()
data_to_plot = pd.concat([mean_rel_gene_change, original_shambhala.mean()], axis=1)
data_to_plot.columns = ['mean_rel_gene_change', 'mean_expression']
sns.scatterplot(data=data_to_plot, x='mean_expression', y='mean_rel_gene_change', alpha=0.1)



In [ ]:
original_shambhala_log = log_transform_by_cohort(original_shambhala, comb_ann_shambhala)

plot_batch_distributions(original_shambhala_log, comb_ann_shambhala, batch_col='COHORT_LABEL', title_prefix="Shambhala original",
                        figsize=(28, 20))

In [ ]:
speed_shambhala_log = log_transform_by_cohort(speed_shambhala, comb_ann_shambhala)

plot_batch_distributions(speed_shambhala_log, comb_ann_shambhala, batch_col='COHORT_LABEL', title_prefix="Shambhala speed",
                        figsize=(28, 20))

In [ ]:
#relative_difference_log = log_transform_by_cohort(relative_difference, comb_ann_shambhala)

plot_batch_distributions(relative_difference.clip(0, 0.25),
                         comb_ann_shambhala, batch_col='COHORT_LABEL', title_prefix="Shambhala relative difference",
                        figsize=(28, 20))

In [ ]:
## Dependency of relative difference on the average expression